# MV-Fashion Dataset - Complete Tutorial

This notebook demonstrates how to download, load, and visualize the MV-Fashion dataset.

**Dataset**: [MV-Fashion on HuggingFace](https://huggingface.co/datasets/MV-Fashion/MV-Fashion)  
**Paper**: [MV-Fashion: Towards Enabling Virtual Try-On and Size Estimation with Multi-View Paired Data (CVPR 2026)](https://arxiv.org/abs/2603.08147)

## Overview

MV-Fashion is a multi-view video dataset for virtual try-on and size estimation research:
- **81 subjects** with diverse body types
- **332 outfits** with multiple garments
- **2,357 sequences** of subjects wearing outfits
- **76 cameras** (60 RPi + 16 Bolt with depth)
- **52M+ frames** total

## What You'll Learn

1. How to download specific parts of the dataset from HuggingFace
2. How to unzip video archives
3. How to navigate the dataset structure
4. How to load and visualize multi-view frames
5. How to access garment metadata and images
6. How to download and visualize segmentation masks (foreground and garment)
7. How to work with camera calibration
8. How to analyze dataset statistics

## Setup

First, let's install required packages and set up the environment.

In [ ]:
# Install required packages
!pip install -q huggingface_hub pillow matplotlib numpy opencv-python

In [ ]:
import sys
from pathlib import Path

# The notebook lives in utils/, so add the current directory to the path
tools_dir = Path.cwd()
sys.path.insert(0, str(tools_dir))

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Import dataset helpers directly (no unified loader class)
from paths import (
    iter_sequences, get_sequence_path, get_subjects, get_outfits,
    get_layers, get_sequences,
)
from metadata import (
    get_subject_info, get_camera_K, get_camera_distortion, load_intrinsics,
)
from garments import get_all_garments_in_outfit, get_garment_description
from statistics import (
    count_subjects, count_total_outfits, count_total_sequences,
    count_total_garments, count_garments_by_type, count_garments_by_fabric,
)
from validation import validate_subject_metadata

print("✓ Setup complete")


## 1. Download the Dataset

The dataset is hosted on HuggingFace and requires access approval. Follow these steps:

### Step 1: Request Access

1. Go to https://huggingface.co/datasets/MV-Fashion/MV-Fashion
2. Follow the instructions in the request form
3. Wait for approval (usually a few business days)

### Step 2: Set Up Authentication

Once approved, set your HuggingFace token:

In [ ]:
import os

# Option 1: Set your token directly (replace with your actual token)
# os.environ['HF_TOKEN'] = 'your_token_here'

# Option 2: Load from a .env file in the project root or current directory
try:
    from dotenv import load_dotenv
    load_dotenv()  # Looks for .env in CWD and parent directories
    if not os.environ.get('HF_TOKEN'):
        load_dotenv(Path.cwd().parent / '.env')  # Try the project root
except ImportError:
    pass

# Check if the token is now available
if os.environ.get('HF_TOKEN'):
    print(f"✓ Token loaded: {os.environ['HF_TOKEN'][:10]}...{os.environ['HF_TOKEN'][-4:]}")
else:
    print("⚠ HF_TOKEN not set. Request access at https://huggingface.co/datasets/MV-Fashion/MV-Fashion")
    print("  then set it via the HF_TOKEN environment variable or a .env file.")


### Step 3: Download the Dataset

You can download the entire dataset or specific parts. The full dataset is ~11.5 TB, so you may want to start with a subset.

In [ ]:
from huggingface_hub import snapshot_download, HfApi

REPO_ID = "MV-Fashion/MV-Fashion"
OUTPUT_DIR = Path(os.environ.get("MV_FASHION_DATASET_ROOT", "./mv_fashion_dataset"))

# Check access
api = HfApi(token=os.environ['HF_TOKEN'])
try:
    api.dataset_info(REPO_ID)
    print("✓ Access granted to MV-Fashion dataset")
except Exception as e:
    print(f"✗ Access denied: {e}")
    print("Please request access at https://huggingface.co/datasets/MV-Fashion/MV-Fashion")

In [ ]:
# Download garment data for subject 1, outfit 1 only
print("Downloading garment data for subject_0001/outfit_1...")

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    token=os.environ['HF_TOKEN'],
    local_dir=OUTPUT_DIR,
    allow_patterns=[
        "info.csv",
        "cameras/**",
        "garments/descriptions/subject_0001.json",
        "garments/measurements/0001_outfit_1.json",
        "garments/styles/0001_styling_labels.json",
        "garments/images/subject_0001/outfit_1/**",
        "garments/masks/subject_0001/outfit_1/**",
    ],
    max_workers=4,
)

print("✓ Garment data downloaded")

In [ ]:
# Download videos for subject 1, outfit 1, layer 1, sequences 1 and 2 only
print("Downloading videos for subject_0001/outfit_1/layer_1/sequence_1 and sequence_2...")

for seq_num in [1, 2]:
    sequence_str = f"sequence_{seq_num}"
    pattern = f"videos/subject_0001/outfit_1/layer_1/{sequence_str}/**"
    print(f"  Downloading {sequence_str}...")
    
    snapshot_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        token=os.environ['HF_TOKEN'],
        local_dir=OUTPUT_DIR,
        allow_patterns=[pattern],
        max_workers=4,
    )

print("✓ Videos downloaded")

## 2. Unzip Video Archives

Video files are stored as zip archives. Each sequence contains:
- `rpi.zip`: 60 RPi camera videos (~2.4 GB per sequence)
- `bolt.zip`: 8 Bolt camera videos with depth (~1.8 GB per sequence)

Let's unzip a sequence to work with the frames.

In [ ]:
import zipfile

def unzip_archive(archive_path, output_dir=None, overwrite=False):
    """Unzip a single archive."""
    extract_dir = output_dir or archive_path.parent
    
    with zipfile.ZipFile(archive_path, 'r') as zf:
        members = zf.namelist()
        print(f"  Extracting {len(members)} files...")
        
        for member in members:
            target_path = extract_dir / member
            
            if target_path.exists() and not overwrite:
                continue
            
            if member.endswith('/'):
                target_path.mkdir(parents=True, exist_ok=True)
            else:
                target_path.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(member) as src, open(target_path, 'wb') as dst:
                    dst.write(src.read())
    
    print(f"  ✓ Extracted to {extract_dir}")

# Unzip sequences 1 and 2
for seq_num in [1, 2]:
    seq_path = OUTPUT_DIR / f"videos/subject_0001/outfit_1/layer_1/sequence_{seq_num}"
    print(f"\nUnzipping sequence_{seq_num}...")
    
    print("  Unzipping RPi videos...")
    unzip_archive(seq_path / "rpi.zip")
    
    print("  Unzipping Bolt videos...")
    unzip_archive(seq_path / "bolt.zip")

print("\n✓ Videos unzipped")

## 3. Navigate the Dataset

Now let's explore the dataset structure using the helper functions.

In [ ]:
# Get basic statistics using the helper modules directly
print("Dataset Overview:")
print(f"  Subjects: {count_subjects()}")
print(f"  Outfits: {count_total_outfits()}")
print(f"  Sequences: {count_total_sequences()}")
print(f"  Garments: {count_total_garments()}")
print(f"  Cameras: {len(load_intrinsics())}")


In [ ]:
# List all subjects
subjects = get_subjects()
print(f"Total subjects: {len(subjects)}")
print(f"First 10 subjects: {subjects[:10]}")


In [ ]:
# Navigate hierarchy for a specific subject
subject = "subject_0001"

print(f"\n{subject}:")
print(f"  Info: {get_subject_info(subject)}")

outfits = get_outfits(subject)
print(f"  Outfits: {outfits}")

for outfit in outfits:
    layers = get_layers(subject, outfit)
    print(f"    {outfit}: {len(layers)} layer(s)")
    
    for layer in layers:
        sequences = get_sequences(subject, outfit, layer)
        print(f"      {layer}: {len(sequences)} sequence(s)")


## 4. Visualize Multi-View Frames

Let's load and visualize frames from multiple cameras.

In [ ]:
def load_frame(sequence_dir, camera, frame_idx):
    """Load a single frame from extracted videos."""
    frame_path = sequence_dir / camera / f"{frame_idx:05d}.avif"
    
    if not frame_path.exists():
        return None
    
    try:
        img = Image.open(frame_path)
        return np.array(img)
    except Exception as e:
        print(f"Error loading {frame_path}: {e}")
        return None

# Load frames from first sequence
seq_dir = OUTPUT_DIR / "videos/subject_0001/outfit_1/layer_1/sequence_1"
frame_idx = 0

# Load 8 RPi cameras
rpi_cameras = [f"rpi_{i:02d}" for i in range(1, 9)]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle(f"Multi-View Frames - Frame {frame_idx}", fontsize=14)

for idx, cam in enumerate(rpi_cameras):
    row, col = idx // 4, idx % 4
    ax = axes[row, col]
    
    img = load_frame(seq_dir, cam, frame_idx)
    
    if img is not None:
        ax.imshow(img)
        ax.set_title(cam, fontsize=10)
    else:
        ax.text(0.5, 0.5, "Not found", ha="center", va="center")
        ax.set_title(cam, fontsize=10)
    
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Load Bolt camera frames (RGB + Depth)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle(f"Bolt Camera Frames - Frame {frame_idx}", fontsize=14)

bolt_cameras = [f"bolt_{i}" for i in range(1, 5)]

for idx, cam in enumerate(bolt_cameras):
    # RGB
    rgb_path = seq_dir / cam / "rgb" / f"{frame_idx:05d}.avif"
    if rgb_path.exists():
        img = Image.open(rgb_path)
        axes[0, idx].imshow(img)
        axes[0, idx].set_title(f"{cam} RGB", fontsize=10)
    else:
        axes[0, idx].text(0.5, 0.5, "Not found", ha="center", va="center")
        axes[0, idx].set_title(f"{cam} RGB", fontsize=10)
    axes[0, idx].axis("off")
    
    # Depth
    depth_path = seq_dir / cam / "depth" / f"{frame_idx:05d}.npz"
    if depth_path.exists():
        depth_data = np.load(depth_path)
        depth = depth_data[list(depth_data.keys())[0]]
        axes[1, idx].imshow(depth, cmap="viridis")
        axes[1, idx].set_title(f"{cam} Depth", fontsize=10)
    else:
        axes[1, idx].text(0.5, 0.5, "Not found", ha="center", va="center")
        axes[1, idx].set_title(f"{cam} Depth", fontsize=10)
    axes[1, idx].axis("off")

plt.tight_layout()
plt.show()

## 5. Access Garment Metadata

The dataset includes rich metadata about garments: descriptions, measurements, and styling labels.

In [ ]:
# Get garment information for an outfit
subject = "subject_0001"
outfit = "outfit_1"

garments = get_all_garments_in_outfit(subject, outfit)

print(f"Garments in {subject}/{outfit}:\n")

for garment in garments:
    cloth_num = garment.get("cloth_number")
    garment_type = garment.get("type")
    fabric = garment.get("fabric")
    layer = garment.get("layer")
    
    print(f"Cloth {cloth_num}:")
    print(f"  Type: {garment_type}")
    print(f"  Fabric: {fabric}")
    print(f"  Layer: {layer}")
    
    measurements = garment.get("measurements", {})
    if measurements:
        print(f"  Measurements:")
        for key, value in list(measurements.items())[:5]:
            print(f"    {key}: {value}")
    
    print()

# --- Style labels ---
# Each sequence also has styling labels (button_zip_style, tucking_style,
# long_sleeve_style, hooding_style, ...) keyed by layer.
from garments import get_styling_label

for seq_num in [1, 2]:
    sequence = f"sequence_{seq_num}"
    style = get_styling_label(subject, outfit, "layer_1", sequence)
    if not style:
        continue
    print(f"Style labels for {sequence}:")
    for layer_key, attrs in style.items():
        print(f"  {layer_key}:")
        for attr, info in attrs.items():
            if isinstance(info, dict):
                print(f"    {attr}: {info.get('description', info)}")
            else:
                print(f"    {attr}: {info}")
    print()


In [ ]:
# Get garment descriptions
desc = get_garment_description(subject, outfit, "cloth_1")

if desc:
    print(f"Cloth 1 description:\n{desc}")

## 6. Visualize Garment Images

The dataset includes flat catalogue images of each garment (front and back views).

In [ ]:
from paths import get_cloth_images
from collections import defaultdict

# Get garment images
cloth_images = get_cloth_images(subject, outfit)
print(f"Found {len(cloth_images)} garment images")

# Group by cloth number so we can render every (cloth, view, capture) image
grouped = defaultdict(list)
for img in cloth_images:
    grouped[img["cloth_number"]].append(img)
for k in grouped:
    grouped[k].sort(key=lambda d: (d["view"], d["capture_index"]))

if grouped:
    num_cloths = len(grouped)
    # Always render up to 4 columns per cloth: front_1, front_2, back_1, back_2
    fig, axes = plt.subplots(num_cloths, 4, figsize=(16, 4 * num_cloths))
    if num_cloths == 1:
        axes = axes.reshape(1, -1)
    fig.suptitle(f"{subject}/{outfit} - Garment Catalogue Images", fontsize=14)
    
    for row_idx, cloth_num in enumerate(sorted(grouped.keys())):
        imgs = grouped[cloth_num]
        garment_type = next((g.get("type", "unknown") for g in garments if g.get("cloth_number") == cloth_num), "unknown")
        for col_idx, key in enumerate([("front", 1), ("front", 2), ("back", 1), ("back", 2)]):
            view, cap = key
            ax = axes[row_idx, col_idx]
            match = next((im for im in imgs if im["view"] == view and im["capture_index"] == cap), None)
            if match and match["path"].exists():
                ax.imshow(Image.open(match["path"]))
                ax.set_title(f"Cloth {cloth_num} ({garment_type})\n{view}_{cap}", fontsize=10)
            else:
                ax.text(0.5, 0.5, "Not found", ha="center", va="center")
                ax.set_title(f"Cloth {cloth_num}\n{view}_{cap} missing", fontsize=10)
            ax.axis("off")
    
    plt.tight_layout()
    plt.show()
else:
    print("No cloth images to display.")


## 7. Visualize Segmentation Masks

The dataset includes two types of segmentation masks:
- **Foreground masks**: Binary masks separating the person from the background
- **Garment masks**: Per-garment segmentation masks for each cloth item

Let's download and visualize both types of masks.

In [ ]:
# Download segmentation masks for subject 1, outfit 1
print("Downloading segmentation masks for subject_0001/outfit_1...")

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    token=os.environ['HF_TOKEN'],
    local_dir=OUTPUT_DIR,
    allow_patterns=[
        "annotations/masks/foreground/subject_0001/outfit_1/**",
        "annotations/masks/garment/subject_0001/outfit_1/**",
    ],
    max_workers=4,
)

print("✓ Segmentation masks downloaded")

In [ ]:
from paths import get_foreground_mask_dir, get_garment_mask_dir

# Unzip foreground masks for both sequences
for seq_num in [1, 2]:
    fg_mask_archive = OUTPUT_DIR / f"annotations/masks/foreground/subject_0001/outfit_1/layer_1/sequence_{seq_num}/rpi.zip"
    if fg_mask_archive.exists():
        print(f"Unzipping foreground masks for sequence_{seq_num}...")
        unzip_archive(fg_mask_archive)

# Unzip garment masks for both sequences
for seq_num in [1, 2]:
    garment_mask_archive = OUTPUT_DIR / f"annotations/masks/garment/subject_0001/outfit_1/layer_1/sequence_{seq_num}/rpi.zip"
    if garment_mask_archive.exists():
        print(f"Unzipping garment masks for sequence_{seq_num}...")
        unzip_archive(garment_mask_archive)

print("\n✓ Masks unzipped")

In [ ]:
# Visualize foreground masks: frame, binary mask, and frame with overlay.
fg_mask_dir = get_foreground_mask_dir("subject_0001", "outfit_1")
seq_dir = OUTPUT_DIR / "videos/subject_0001/outfit_1/layer_1/sequence_1"

print(f"Foreground mask directory: {fg_mask_dir}")

fg_mask_path = fg_mask_dir / "layer_1" / "sequence_1" / "rpi_01"
if fg_mask_path.exists():
    mask_files = sorted(fg_mask_path.glob("*.png"))
    print(f"Found {len(mask_files)} foreground masks for rpi_01")
    
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    fig.suptitle("Foreground Masks: Frame, Binary Mask, Overlay", fontsize=14)
    alpha = 0.45
    overlay_color = np.array([0, 255, 0], dtype=np.uint8)  # green for person
    
    for idx in range(min(4, len(mask_files))):
        frame_idx = idx
        frame = load_frame(seq_dir, "rpi_01", frame_idx)
        mask = np.array(Image.open(mask_files[idx]))
        # Resize mask to match frame resolution (masks in the dataset should
        # already be frame-resolution, but resize defensively in case they
        # differ).
        if frame is not None and mask.shape[:2] != frame.shape[:2]:
            mask_img = Image.fromarray(mask).resize(
                (frame.shape[1], frame.shape[0]), Image.NEAREST
            )
            mask = np.array(mask_img)
        
        # Row 0: original frame
        if frame is not None:
            axes[0, idx].imshow(frame)
        axes[0, idx].set_title(f"Frame {frame_idx}", fontsize=10)
        axes[0, idx].axis("off")
        
        # Row 1: binary mask
        axes[1, idx].imshow(mask, cmap="gray")
        axes[1, idx].set_title(f"Foreground Mask", fontsize=10)
        axes[1, idx].axis("off")
        
        # Row 2: overlay (semi-transparent green where the person is)
        if frame is not None:
            overlay = frame.copy()
            mask_bool = mask > 127
            overlay[mask_bool] = (overlay[mask_bool] * (1 - alpha) + overlay_color * alpha).astype(np.uint8)
            axes[2, idx].imshow(overlay)
        axes[2, idx].set_title(f"Overlay (alpha={alpha})", fontsize=10)
        axes[2, idx].axis("off")
    
    plt.tight_layout()
    plt.show()
else:
    print(f"Foreground mask directory not found: {fg_mask_path}")


In [ ]:
# Visualize garment masks: frame, foreground mask, garment mask, and overlays
garment_mask_dir = get_garment_mask_dir("subject_0001", "outfit_1")
print(f"Garment mask directory: {garment_mask_dir}")

garment_mask_path = garment_mask_dir / "layer_1" / "sequence_1" / "cloth_1" / "rpi_01"
if garment_mask_path.exists():
    garment_mask_files = sorted(garment_mask_path.glob("*.png"))
    print(f"Found {len(garment_mask_files)} garment masks for cloth_1, rpi_01")
    
    fg_mask_path = fg_mask_dir / "layer_1" / "sequence_1" / "rpi_01"
    fg_mask_files = sorted(fg_mask_path.glob("*.png")) if fg_mask_path.exists() else []
    
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    fig.suptitle("Garment Masks (cloth_1): Frame, Foreground, Garment, Overlay", fontsize=14)
    
    alpha_val = 0.45
    overlay_color = np.array([255, 80, 80], dtype=np.uint8)  # red for cloth_1
    
    for idx in range(min(4, len(garment_mask_files))):
        frame_idx = idx
        frame = load_frame(seq_dir, "rpi_01", frame_idx)
        fg_mask = np.array(Image.open(fg_mask_files[idx])) if idx < len(fg_mask_files) else None
        garment_mask = np.array(Image.open(garment_mask_files[idx]))
        # Resize both masks to the frame resolution (defensive)
        if frame is not None:
            if garment_mask.shape[:2] != frame.shape[:2]:
                garment_mask = np.array(Image.fromarray(garment_mask).resize(
                    (frame.shape[1], frame.shape[0]), Image.NEAREST))
            if fg_mask is not None and fg_mask.shape[:2] != frame.shape[:2]:
                fg_mask = np.array(Image.fromarray(fg_mask).resize(
                    (frame.shape[1], frame.shape[0]), Image.NEAREST))
        
        axes[0, idx].imshow(frame if frame is not None else np.zeros_like(garment_mask))
        axes[0, idx].set_title(f"Frame {frame_idx}", fontsize=10)
        axes[0, idx].axis("off")
        
        if fg_mask is not None:
            axes[1, idx].imshow(fg_mask, cmap="gray")
        axes[1, idx].set_title("Foreground Mask", fontsize=10)
        axes[1, idx].axis("off")
        
        axes[2, idx].imshow(garment_mask, cmap="gray")
        axes[2, idx].set_title("Garment Mask (cloth_1)", fontsize=10)
        axes[2, idx].axis("off")
        
        if frame is not None:
            overlay = frame.copy()
            mask_bool = garment_mask > 127
            overlay[mask_bool] = (overlay[mask_bool] * (1 - alpha_val) + overlay_color * alpha_val).astype(np.uint8)
            axes[3, idx].imshow(overlay)
        axes[3, idx].set_title(f"Overlay (alpha={alpha_val})", fontsize=10)
        axes[3, idx].axis("off")
    
    plt.tight_layout()
    plt.show()
else:
    print(f"Garment mask directory not found: {garment_mask_path}")


In [ ]:
# Visualize all garment masks for a single frame, overlaid with a different color per cloth.
# Only garments whose `layer` is <= the sequence's layer are plotted, so for a
# sequence stored in layer_1/sequence_X we only see layer-1 clothes; for a
# sequence in layer_2 we would see layers 1 and 2; etc.
frame_idx = 0
sequence_layer = 1  # the sequence we are plotting lives in layer_1/
garment_mask_base = garment_mask_dir / f"layer_{sequence_layer}" / "sequence_1"

if garment_mask_base.exists():
    # Filter the outfit's garments to those with layer <= sequence_layer
    eligible_garments = [g for g in garments if (g.get("layer") or 99) <= sequence_layer]
    eligible_cloth_nums = sorted(g.get("cloth_number") for g in eligible_garments if g.get("cloth_number") is not None)
    print(f"Sequence layer: {sequence_layer}")
    print(f"Eligible clothes (layer <= {sequence_layer}): {eligible_cloth_nums}")
    
    # Predefined color per cloth index for consistent rendering
    palette = [
        np.array([255, 80, 80], dtype=np.uint8),    # red
        np.array([80, 200, 80], dtype=np.uint8),   # green
        np.array([80, 130, 255], dtype=np.uint8),  # blue
        np.array([255, 200, 80], dtype=np.uint8),  # orange
        np.array([220, 80, 220], dtype=np.uint8),  # magenta
        np.array([80, 220, 220], dtype=np.uint8),  # cyan
        np.array([255, 130, 200], dtype=np.uint8), # pink
    ]
    alpha_val = 0.45
    
    # Two rows: row 0 = individual binary masks, row 1 = combined overlay on frame
    num_cloths = len(eligible_cloth_nums)
    fig, axes = plt.subplots(2, num_cloths, figsize=(4 * num_cloths, 8))
    fig.suptitle(f"All Garment Masks - Frame {frame_idx} (layer<={sequence_layer})", fontsize=14)
    if num_cloths == 1:
        axes = axes.reshape(2, 1)
    
    frame = load_frame(seq_dir, "rpi_01", frame_idx)
    
    for col_idx, cloth_num in enumerate(eligible_cloth_nums):
        cloth_dir = garment_mask_base / f"cloth_{cloth_num}" / "rpi_01"
        garment = next((g for g in eligible_garments if g.get("cloth_number") == cloth_num), {})
        garment_type = garment.get("type", "unknown")
        
        # Row 0: binary mask
        mask = None
        if cloth_dir.exists():
            mask_files = sorted(cloth_dir.glob("*.png"))
            if frame_idx < len(mask_files):
                mask = np.array(Image.open(mask_files[frame_idx]))
                if frame is not None and mask.shape[:2] != frame.shape[:2]:
                    mask = np.array(Image.fromarray(mask).resize(
                        (frame.shape[1], frame.shape[0]), Image.NEAREST))
                axes[0, col_idx].imshow(mask, cmap="gray")
        axes[0, col_idx].set_title(f"cloth_{cloth_num} ({garment_type})", fontsize=10)
        axes[0, col_idx].axis("off")
        
        # Row 1: frame with this cloth's mask overlaid in its palette color
        if frame is not None and mask is not None:
            overlay = frame.copy()
            mask_bool = mask > 127
            color = palette[col_idx % len(palette)]
            overlay[mask_bool] = (overlay[mask_bool] * (1 - alpha_val) + color * alpha_val).astype(np.uint8)
            axes[1, col_idx].imshow(overlay)
        axes[1, col_idx].set_title(f"Overlay cloth_{cloth_num}", fontsize=10)
        axes[1, col_idx].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    # Also show a single image with every cloth's mask painted in its own color
    if frame is not None:
        combined = frame.copy()
        for col_idx, cloth_num in enumerate(eligible_cloth_nums):
            cloth_dir = garment_mask_base / f"cloth_{cloth_num}" / "rpi_01"
            if not cloth_dir.exists():
                continue
            mask_files = sorted(cloth_dir.glob("*.png"))
            if frame_idx >= len(mask_files):
                continue
            mask = np.array(Image.open(mask_files[frame_idx]))
            if mask.shape[:2] != frame.shape[:2]:
                mask = np.array(Image.fromarray(mask).resize(
                    (frame.shape[1], frame.shape[0]), Image.NEAREST))
            color = palette[col_idx % len(palette)]
            mask_bool = mask > 127
            combined[mask_bool] = (combined[mask_bool] * (1 - alpha_val) + color * alpha_val).astype(np.uint8)
        plt.figure(figsize=(8, 12))
        plt.imshow(combined)
        plt.title(f"Combined Overlay (all clothes, layer<={sequence_layer})")
        plt.axis("off")
        plt.show()
else:
    print(f"Garment mask base directory not found: {garment_mask_base}")


## 8. Work with Camera Calibration

The dataset provides camera intrinsics and extrinsics for all cameras.

In [ ]:
# Get camera intrinsics
camera = "rpi_01"
K = get_camera_K(camera, use_undistorted=True)

print(f"Camera {camera} intrinsics (undistorted):")
print(f"K = \n{K}")

intrinsics = load_intrinsics().get(camera)
if intrinsics is not None:
    print(f"\nImage shape: {intrinsics.get('image_shape')}")
    print(f"Reprojection error: {intrinsics.get('reproj_error_rms'):.4f}")


In [ ]:
# Undistort a frame using the camera intrinsics.
# Raw frames from the RPi/Bolt cameras are lens-distorted. The dataset provides
# the original camera matrix K and the distortion coefficients `dist`. It also
# ships a `K_new` value, but that is the rectified intrinsics for the
# dataset's stereo / multi-view pipeline - feeding it directly to
# `cv2.undistort` shifts the principal point off-centre and produces a
# visibly off-centre rectified image. For monocular undistortion we instead
# call `cv2.getOptimalNewCameraMatrix` to derive a K_new that keeps the
# principal point at the image centre.
import cv2
from metadata import get_camera_distortion
from visualize_dataset import undistort_frame

camera = "rpi_01"
K = get_camera_K(camera, use_undistorted=False)       # original camera matrix
dist = get_camera_distortion(camera)                 # distortion coefficients
K_new_dataset = get_camera_K(camera, use_undistorted=True)  # dataset's K_new

frame = load_frame(seq_dir, camera, 0)

if frame is not None and K is not None and dist is not None and K_new_dataset is not None:
    h, w = frame.shape[:2]
    K_new_optimal = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), 1, (w, h))
    if isinstance(K_new_optimal, tuple):
        K_new_optimal = K_new_optimal[0]
    print(f"Camera {camera}:")
    print(f"  K principal point:  ({K[0,2]:.1f}, {K[1,2]:.1f})")
    print(f"  K_new (dataset) pp: ({K_new_dataset[0,2]:.1f}, {K_new_dataset[1,2]:.1f})")
    print(f"  K_new (optimal) pp: ({K_new_optimal[0,2]:.1f}, {K_new_optimal[1,2]:.1f})")
    print(f"  Image center:       ({w/2:.1f}, {h/2:.1f})")
    
    undistorted = undistort_frame(frame, camera)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f"Undistortion - {camera} (Frame 0)", fontsize=14)
    axes[0].imshow(frame)
    axes[0].set_title("Original (distorted)")
    axes[0].axis("off")
    axes[1].imshow(undistorted)
    axes[1].set_title("Undistorted (centred)")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Frame or intrinsics not available; run the download/unzip cells first.")


In [ ]:
# Bolt extrinsics: factory-calibrated transformation between the Bolt RGB
# sensor and the Bolt depth sensor on the same device. The dataset provides
# one (R, t) pair per Bolt camera (bolt_1 ... bolt_8), describing the pose
# of the depth frame in the RGB frame: a 3D point P_depth in the depth
# sensor's coordinates maps to P_rgb = R @ P_depth + t in the RGB sensor's
# coordinates. They are calibrated at the factory and not learned from the
# MV-Fashion captures themselves.
from metadata import get_bolt_rotation, get_bolt_translation

bolt = "bolt_1"
R = get_bolt_rotation(bolt)
t = get_bolt_translation(bolt)

if R is None or t is None:
    print(f"No extrinsics for {bolt}; download cameras/extrinsics_bolts.json first.")
else:
    print(f"Bolt {bolt} extrinsics (factory: depth -> RGB):")
    print(f"  R =\n{R}")
    print(f"\n  t (mm) = {t}")
    print(f"  |t|   = {np.linalg.norm(t):.2f} mm")

# Show the difference in sensor resolution between the two streams
intrinsics = load_intrinsics()
intr_rgb = intrinsics.get(bolt)
intr_depth = intrinsics.get(f"{bolt}_depth")
if intr_rgb is not None:
    print(f"\n  RGB sensor:   {intr_rgb.get('image_shape')}  (intrinsics key: {bolt})")
if intr_depth is not None:
    print(f"  Depth sensor: {intr_depth.get('image_shape')}  (intrinsics key: {bolt}_depth)")
else:
    print(f"\n  Depth intrinsics for {bolt}_depth are not in the dataset.")
print("\nUse R, t to project a depth point into the RGB frame, then divide by")
print("the depth value to get pixel coordinates. K_rgb and K_depth (from the")
print("intrinsics JSON) are needed to convert to / from pixel space.")


## 9. Analyze Dataset Statistics

Let's analyze the dataset composition and statistics.

In [ ]:
from statistics import count_garments_by_type, count_garments_by_fabric

# Garment type distribution
type_counts = count_garments_by_type()
top_types = dict(sorted(type_counts.items(), key=lambda x: -x[1])[:10])

print("Top 10 Garment Types:")
for garment_type, count in top_types.items():
    print(f"  {garment_type}: {count}")

In [ ]:
# Visualize statistics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Garment types
axes[0].barh(list(top_types.keys()), list(top_types.values()))
axes[0].set_xlabel("Count")
axes[0].set_title("Top 10 Garment Types")
axes[0].invert_yaxis()

# Garment fabrics
fabric_counts = count_garments_by_fabric()
top_fabrics = dict(sorted(fabric_counts.items(), key=lambda x: -x[1])[:10])

axes[1].barh(list(top_fabrics.keys()), list(top_fabrics.values()), color="orange")
axes[1].set_xlabel("Count")
axes[1].set_title("Top 10 Garment Fabrics")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 10. Iterate Through All Sequences

Here's how to iterate through all sequences in the dataset.

In [ ]:
# Iterate through first 10 sequences
print("First 10 sequences:\n")

for i, (subject, outfit, layer, sequence) in enumerate(iter_sequences()):
    if i >= 10:
        break
    
    seq_path = get_sequence_path(subject, outfit, layer, sequence)
    print(f"{i+1}. {subject}/{outfit}/{layer}/{sequence}")
    print(f"   Path: {seq_path}")
    
    # Check if archives exist
    rpi_exists = (seq_path / "rpi.zip").exists()
    bolt_exists = (seq_path / "bolt.zip").exists()
    print(f"   Archives: RPi={rpi_exists}, Bolt={bolt_exists}")
    print()

## 11. Complete Example: Process a Sequence

Here's a complete example showing how to process a sequence end-to-end.

In [ ]:
def process_sequence(subject, outfit, layer, sequence):
    """Complete example: process a sequence end-to-end."""
    
    print(f"Processing {subject}/{outfit}/{layer}/{sequence}\n")
    
    # 1. Get sequence path
    seq_path = get_sequence_path(subject, outfit, layer, sequence)
    print(f"1. Sequence path: {seq_path}")
    
    # 2. Get subject info
    subject_info = get_subject_info(subject)
    print(f"2. Subject: {subject_info['gender']}, {subject_info['height']}cm, {subject_info['weight']}kg")
    
    # 3. Get garment info
    garments = get_all_garments_in_outfit(subject, outfit)
    print(f"3. Outfit has {len(garments)} garment(s):")
    for g in garments:
        print(f"   - Cloth {g['cloth_number']}: {g['type']} ({g['fabric']})")
    
    # 4. Check video archives
    rpi_archive = seq_path / "rpi.zip"
    bolt_archive = seq_path / "bolt.zip"
    print(f"4. Video archives: RPi={rpi_archive.exists()}, Bolt={bolt_archive.exists()}")
    
    # 5. Load a sample frame
    frame_idx = 0
    frame = load_frame(seq_path, "rpi_01", frame_idx)
    if frame is not None:
        print(f"5. Loaded frame {frame_idx}: shape={frame.shape}")
        
        plt.figure(figsize=(8, 6))
        plt.imshow(frame)
        plt.title(f"Sample frame from rpi_01")
        plt.axis("off")
        plt.show()
    
    print("\n✓ Sequence processing complete")

# Process first sequence
process_sequence("subject_0001", "outfit_1", "layer_1", "sequence_1")

## Summary

In this notebook, you learned how to:

1. **Download specific dataset components** from HuggingFace (garments for subject 1, outfit 1; videos for 2 sequences)
2. **Unzip video archives** to extract multi-view frames
3. **Navigate the dataset** structure using helper functions
4. **Visualize multi-view frames** from RPi and Bolt cameras
5. **Access garment metadata** including descriptions and measurements
6. **Visualize garment images** (flat catalogue views)
7. **Download and visualize segmentation masks** (both foreground and per-garment masks)
8. **Work with camera calibration** data (intrinsics and extrinsics)
9. **Analyze dataset statistics** and composition
10. **Iterate through sequences** programmatically
11. **Process sequences** end-to-end

## Next Steps

- Explore more sequences and subjects
- Use the styling labels for pose analysis
- Combine masks with video frames for training
- Build your own virtual try-on or size estimation models

## Resources

- [Dataset on HuggingFace](https://huggingface.co/datasets/MV-Fashion/MV-Fashion)
- [Paper on arXiv](https://arxiv.org/abs/2603.08147)
- [Project Page](https://hunorlaczko.github.io/MV-Fashion/)
- [GitHub Repository](https://github.com/HunorLaczko/MV-Fashion)